# 2. Datenbereinigung

- TP3 entfernt: identisch mit Reservoirs (r = 1,00).
- Kleine negative Druckwerte auf 0 gesetzt: Sensorrauschen um den Nullpunkt.
- Segmentierung an Datenlücken > 1 min: Die Daten werden in zusammenhängende Segmente getrennt. Berechnungen erfolgen nur innerhalb eines Segments.

In [1]:
import pandas as pd

df = pd.read_parquet("../data/processed/df_raw.parquet")
print(df.shape)

(1516948, 15)


In [2]:
import pandas as pd

# 1. Redundante Spalten entfernen
#    TP3 identisch mit Reservoirs (r = 1,00)
#    Caudal_impulses hochkorreliert mit COMP/MPG → redundant
clean = df.drop(columns=["TP3", "Caudal_impulses"]).copy()

# 2. Kleine negative Druckwerte auf 0 setzen (Sensorrauschen)
PRESSURES = ["TP2", "H1", "DV_pressure", "Reservoirs"]
clean[PRESSURES] = clean[PRESSURES].clip(lower=0)

# 3. Segmentierung an Datenlücken > 1 min
GAP = pd.Timedelta("1min")
gap = clean.index.to_series().diff()
clean["segment"] = (gap > GAP).cumsum().values

print("Lücken > 1 min:", int((gap > GAP).sum()))
print("Segmente:", clean["segment"].nunique())
print("Shape:", clean.shape)
clean.head()

Lücken > 1 min: 331
Segmente: 332
Shape: (1516948, 14)


,TP2,H1,DV_pressure,Reservoirs,Oil_temperature,Motor_current,COMP,DV_eletric,Towers,MPG,LPS,Pressure_switch,Oil_level,segment
timestamp,,,,,,,,,,,,,,
2020-02-01 00:00:00,0.0,9.340,0.0,9.358,53.600,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0
2020-02-01 00:00:10,0.0,9.332,0.0,9.348,53.675,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0
2020-02-01 00:00:19,0.0,9.322,0.0,9.338,53.600,0.0425,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0
2020-02-01 00:00:29,0.0,9.312,0.0,9.328,53.425,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0
2020-02-01 00:00:39,0.0,9.302,0.0,9.318,53.475,0.0400,1.0,0.0,1.0,1.0,0.0,1.0,1.0,0


In [4]:
clean["split"] = "train"
clean.loc["2020-04-17":, "split"] = "test"
clean.to_parquet("../data/processed/df_clean.parquet")
print("Gespeichert:", clean.shape)

Gespeichert: (1516948, 15)
